<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

<h1 align="center"><b>Laboratorio 5 -- Clasificacion con Datos Desbalanceados</b></h1>

</div>

---

<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

Los datos desbalanceados ocurren cuando una clase tiene muchas mas observaciones que otra. En este problema de deteccion de diabetes, el **86.1%** de los registros corresponden a personas sin diabetes y solo el **13.9%** son casos positivos. Este desbalance provoca que los modelos aprendan a favorecer la clase mayoritaria, obteniendo metricas aparentemente altas de **Accuracy** pero con bajo desempeno en la deteccion de diabeticos.

Para corregirlo, se implementan y comparan tecnicas de:
- **Sobremuestreo** (SMOTE, ADASYN, RandomOverSampler)
- **Submuestreo** (RandomUnderSampler, TomekLinks, ENN)
- **Combinaciones** (SMOTETomek, SMOTEENN)
- **Ensambles** (BalancedRandomForest, EasyEnsemble, RUSBoost, BalancedBagging)

Documentacion de referencia: [imbalanced-learn](https://imbalanced-learn.org/stable/index.html)

Instalacion requerida: `pip install imbalanced-learn`

</div>

---

## 0. Librerias

In [5]:
# =================== #
# CARGAR LIBRERIAS    #
# =================== #

import warnings
import pandas as pd
import numpy as np
import os
import joblib
from pathlib import Path

# Visualizacion
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

# Preprocesamiento
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    confusion_matrix, classification_report, ConfusionMatrixDisplay,
    roc_auc_score, precision_score, recall_score, f1_score,
    precision_recall_curve
)

# Modelos base
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

# Tecnicas de balanceo -- imbalanced-learn
from imblearn.over_sampling import RandomOverSampler, SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler, TomekLinks, EditedNearestNeighbours
from imblearn.combine import SMOTETomek, SMOTEENN
from imblearn.ensemble import (
    BalancedRandomForestClassifier,
    EasyEnsembleClassifier,
    RUSBoostClassifier,
    BalancedBaggingClassifier
)
from imblearn.pipeline import Pipeline as ImbPipeline

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")
plt.rcParams.update({"figure.dpi": 120})
SEED = 123
print("Librerias cargadas correctamente.")

Librerias cargadas correctamente.


---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

<h2>1. Comprension del Problema</h2>

**Variable objetivo:** `Diabetes_binary`
- `0` = Sin diabetes
- `1` = Prediabetes o diabetes

**Contexto:** Dataset BRFSS 2015 del CDC con 253.680 adultos estadounidenses. El objetivo es detectar tempranamente la diabetes a partir de 21 indicadores de salud, estilo de vida y factores socioeconomicos.

**Metrica prioritaria:** En deteccion de enfermedades, minimizar los **Falsos Negativos** es critico -- un diabetico no detectado puede sufrir complicaciones graves. Por eso priorizamos **Recall, F1-Score y AUC-ROC** sobre Accuracy.

</div>

---

## 2. Configuracion de rutas

In [15]:
# ================ #
# CONFIGURAR RUTAS #
# ================ #

mainpath = (
    "/workspaces/ml-project_analitica_datosv/"
    "ml-proyecto_analitica_datos/data/raw/dataset_clasificacion"
)
filename = "diabetes_binary_health_indicators_BRFSS2015.csv"
fullpath = os.path.join(mainpath, filename)

MODELS_DIR = Path(
    "/workspaces/ml-project_analitica_datosv/ml-proyecto_analitica_datos/models"
)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

## 3. Lectura de datos

In [16]:
# ================ #
# LECTURA DE DATOS #
# ================ #

data = pd.read_csv(fullpath, sep=",")
pd.set_option("display.max_columns", None)

In [17]:
# ================ #
# VISTA PRELIMINAR #
# ================ #

data.sample(10)

,Diabetes_binary,HighBP,HighChol,CholCheck,BMI,Smoker,Stroke,HeartDiseaseorAttack,PhysActivity,Fruits,Veggies,HvyAlcoholConsump,AnyHealthcare,NoDocbcCost,GenHlth,MentHlth,PhysHlth,DiffWalk,Sex,Age,Education,Income
25589,0.0,0.0,1.0,1.0,25.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,0.0,9.0,6.0,8.0
143205,0.0,0.0,0.0,1.0,30.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,1.0,5.0,0.0,0.0,6.0,6.0,7.0
181683,0.0,0.0,0.0,1.0,27.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,1.0,0.0,3.0,30.0,0.0,0.0,0.0,7.0,5.0,8.0
172122,0.0,1.0,1.0,1.0,38.0,1.0,0.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,3.0,0.0,0.0,1.0,1.0,11.0,4.0,7.0
161692,0.0,1.0,0.0,0.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,5.0,6.0,7.0
5035,0.0,0.0,0.0,1.0,27.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,2.0,20.0,0.0,0.0,0.0,2.0,5.0,4.0
250158,0.0,0.0,0.0,1.0,20.0,1.0,0.0,0.0,1.0,0.0,1.0,1.0,0.0,1.0,2.0,14.0,3.0,0.0,1.0,1.0,5.0,1.0
173631,0.0,0.0,0.0,1.0,32.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,7.0,5.0,7.0
57864,0.0,0.0,0.0,1.0,27.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,2.0,0.0,0.0,0.0,0.0,10.0,6.0,6.0
55259,0.0,0.0,0.0,1.0,26.0,0.0,0.0,0.0,1.0,1.0,1.0,0.0,1.0,0.0,3.0,0.0,0.0,0.0,1.0,12.0,6.0,5.0


In [18]:
data.info()

<class 'pandas.DataFrame'>
RangeIndex: 253680 entries, 0 to 253679
Data columns (total 22 columns):
 #   Column                Non-Null Count   Dtype  
---  ------                --------------   -----  
 0   Diabetes_binary       253680 non-null  float64
 1   HighBP                253680 non-null  float64
 2   HighChol              253680 non-null  float64
 3   CholCheck             253680 non-null  float64
 4   BMI                   253680 non-null  float64
 5   Smoker                253680 non-null  float64
 6   Stroke                253680 non-null  float64
 7   HeartDiseaseorAttack  253680 non-null  float64
 8   PhysActivity          253680 non-null  float64
 9   Fruits                253680 non-null  float64
 10  Veggies               253680 non-null  float64
 11  HvyAlcoholConsump     253680 non-null  float64
 12  AnyHealthcare         253680 non-null  float64
 13  NoDocbcCost           253680 non-null  float64
 14  GenHlth               253680 non-null  float64
 15  MentHlth   

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 4. Exploracion Inicial -- Desbalanceo de Clases

</div>

In [19]:
# ================================= #
# DISTRIBUCION DE LA CLASE OBJETIVO #
# ================================= #

counts = data["Diabetes_binary"].value_counts()
pcts   = data["Diabetes_binary"].value_counts(normalize=True) * 100

fig = px.bar(
    x=["Sin diabetes (0)", "Diabetes/Prediabetes (1)"],
    y=counts.values,
    text=[f"{v:,}<br>({p:.1f}%)" for v, p in zip(counts.values, pcts.values)],
    color=["Sin diabetes (0)", "Diabetes/Prediabetes (1)"],
    color_discrete_sequence=["#4C9BE8", "#E87D4C"],
    title="Distribucion de la Variable Objetivo -- Diabetes_binary",
    template="simple_white"
)
fig.update_traces(textposition="outside")
fig.update_layout(title_x=0.5, showlegend=False,
                  yaxis_title="Numero de registros", xaxis_title="Clase")
fig.show()

print(f"Clase 0 (sin diabetes):       {counts[0]:>7,}  ({pcts[0]:.1f}%)")
print(f"Clase 1 (diabetes/prediab.):  {counts[1]:>7,}  ({pcts[1]:.1f}%)")
print(f"Razon de desbalanceo: {counts[0]/counts[1]:.1f}:1")

Clase 0 (sin diabetes):       218,334  (86.1%)
Clase 1 (diabetes/prediab.):   35,346  (13.9%)
Razon de desbalanceo: 6.2:1


In [20]:
# ===================== #
# VALORES FALTANTES     #
# ===================== #

missing = data.isnull().sum()
if missing.sum() == 0:
    print("No hay valores faltantes.")
else:
    print(missing[missing > 0])

data.describe().T.round(2)

No hay valores faltantes.


,count,mean,std,min,25%,50%,75%,max
Diabetes_binary,253680.0,0.14,0.35,0.0,0.0,0.0,0.0,1.0
HighBP,253680.0,0.43,0.49,0.0,0.0,0.0,1.0,1.0
HighChol,253680.0,0.42,0.49,0.0,0.0,0.0,1.0,1.0
CholCheck,253680.0,0.96,0.19,0.0,1.0,1.0,1.0,1.0
BMI,253680.0,28.38,6.61,12.0,24.0,27.0,31.0,98.0
Smoker,253680.0,0.44,0.50,0.0,0.0,0.0,1.0,1.0
Stroke,253680.0,0.04,0.20,0.0,0.0,0.0,0.0,1.0
HeartDiseaseorAttack,253680.0,0.09,0.29,0.0,0.0,0.0,0.0,1.0
PhysActivity,253680.0,0.76,0.43,0.0,1.0,1.0,1.0,1.0
Fruits,253680.0,0.63,0.48,0.0,0.0,1.0,1.0,1.0


In [22]:
# ========================= #
# CORRELACION CON OBJETIVO  #
# ========================= #

corr = data.corr()["Diabetes_binary"].drop("Diabetes_binary").sort_values()

fig = px.bar(
    x=corr.values,
    y=corr.index,
    orientation="h",
    color=corr.values,
    color_continuous_scale="RdBu_r",
    title="Correlacion de cada variable con Diabetes_binary",
    template="simple_white"
)
fig.update_layout(title_x=0.5, height=600,
                  xaxis_title="Correlacion de Pearson", yaxis_title="")
fig.show()

---
## 5. Preparacion de los Datos

In [23]:
# ========================= #
# SELECCION DE VARIABLES    #
# ========================= #

target = "Diabetes_binary"

X = data.drop(columns=[target])
y = data[target]

print(f"X shape: {X.shape}")
print(f"y distribucion:\n{y.value_counts()}")

X shape: (253680, 21)
y distribucion:
Diabetes_binary
0.0    218334
1.0     35346
Name: count, dtype: int64


In [24]:
# ================================= #
# VARIABLES NUMERICAS Y CATEGORICAS #
# ================================= #

# Variables con rango amplio que se benefician del escalamiento
numeric_features = ["BMI", "MentHlth", "PhysHlth"]

# Variables binarias y ordinales (no requieren escalamiento)
ordinal_features = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk",
    "Sex", "GenHlth", "Age", "Education", "Income"
]

print(f"Variables numericas ({len(numeric_features)}): {numeric_features}")
print(f"Variables ordinales/binarias ({len(ordinal_features)}): {ordinal_features}")

Variables numericas (3): ['BMI', 'MentHlth', 'PhysHlth']
Variables ordinales/binarias (18): ['HighBP', 'HighChol', 'CholCheck', 'Smoker', 'Stroke', 'HeartDiseaseorAttack', 'PhysActivity', 'Fruits', 'Veggies', 'HvyAlcoholConsump', 'AnyHealthcare', 'NoDocbcCost', 'DiffWalk', 'Sex', 'GenHlth', 'Age', 'Education', 'Income']


In [25]:
# ================================= #
# PREPROCESAMIENTO CON ESCALAMIENTO #
# ================================= #

numeric_transformer_scaled = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

passthrough_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

preprocessor_scaled = ColumnTransformer(transformers=[
    ("num_scaled", numeric_transformer_scaled, numeric_features),
    ("ordinal",    passthrough_transformer,    ordinal_features)
])

# ================================= #
# PREPROCESAMIENTO SIN ESCALAMIENTO #
# ================================= #

all_features = numeric_features + ordinal_features

preprocessor_no_scaled = ColumnTransformer(transformers=[
    ("pass", SimpleImputer(strategy="median"), all_features)
])

print("Preprocesadores definidos.")

Preprocesadores definidos.


In [26]:
# ================ #
# TRAIN TEST SPLIT #
# ================ #

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=SEED
)

print(f"Train: {X_train.shape[0]:,} muestras  |  Test: {X_test.shape[0]:,} muestras")
print(f"Proporcion clase 1 -- Train: {y_train.mean():.3f}  |  Test: {y_test.mean():.3f}")

Train: 202,944 muestras  |  Test: 50,736 muestras
Proporcion clase 1 -- Train: 0.139  |  Test: 0.139


---
## 6. Funcion de Evaluacion

In [27]:
# ===================== #
# FUNCION DE EVALUACION #
# ===================== #

def evaluate_model(model, model_name):
    print(f"=== {model_name} ===")

    # Entrenamiento
    model.fit(X_train, y_train)

    # Predicciones
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1] if hasattr(model, "predict_proba") else None

    # Classification report
    print("\nClassification Report\n")
    print(classification_report(y_test, y_pred,
          target_names=["Sin diabetes", "Diabetes"]))

    # Matriz de confusion -- Plotly
    cm = confusion_matrix(y_test, y_pred)
    vp, fp, fn, vn = cm[1,1], cm[0,1], cm[1,0], cm[0,0]

    cm_df = pd.DataFrame(
        [[vp, fp], [fn, vn]],
        index=["Predicho Positivo", "Predicho Negativo"],
        columns=["Real Positivo", "Real Negativo"]
    )
    labels = [
        [f"VP<br>{vp}", f"FP<br>{fp}"],
        [f"FN<br>{fn}", f"VN<br>{vn}"]
    ]

    fig = px.imshow(
        cm_df, text_auto=False,
        color_continuous_scale=[[0.0,"#7f0000"],[0.5,"#1a1a1a"],[1.0,"#006400"]],
        aspect="auto", title=f"Matriz de Confusion - {model_name}"
    )
    for i in range(2):
        for j in range(2):
            fig.add_annotation(
                x=cm_df.columns[j], y=cm_df.index[i],
                text=labels[i][j], showarrow=False,
                font=dict(size=22, color="white")
            )
    fig.update_layout(width=700, height=500, title_x=0.5,
                      template="plotly_dark", coloraxis_showscale=False)
    fig.update_xaxes(title="Clases reales", side="top")
    fig.update_yaxes(title="Clases predichas")
    fig.show()

    # Metricas
    metrics = {
        "Model":     model_name,
        "Accuracy":  round((y_pred == y_test).mean(), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_test, y_pred), 4),
        "F1":        round(f1_score(y_test, y_pred), 4),
    }
    if y_prob is not None:
        metrics["ROC_AUC"] = round(roc_auc_score(y_test, y_prob), 4)

    metrics_df = pd.DataFrame([metrics])
    print("\nMETRICAS\n")
    display(metrics_df.style.hide(axis="index"))
    return metrics

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 7. Modelo Base -- Referencia sin balanceo

Se entrena un modelo de Regresion Logistica sin ninguna tecnica de balanceo. Sus metricas sirven como **linea base** para medir el impacto de cada tecnica.

</div>

In [14]:
# ================================= #
# MODELO BASE - REGRESION LOGISTICA #
# ================================= #

baseline_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(max_iter=500, random_state=SEED))
])

metrics_baseline = evaluate_model(baseline_model, "Baseline -- Sin balanceo")

=== Baseline -- Sin balanceo ===

Classification Report

              precision    recall  f1-score   support

Sin diabetes       0.88      0.98      0.92     43667
    Diabetes       0.53      0.15      0.23      7069

    accuracy                           0.86     50736
   macro avg       0.70      0.56      0.58     50736
weighted avg       0.83      0.86      0.83     50736




METRICAS



Model,Accuracy,Precision,Recall,F1,ROC_AUC
Baseline -- Sin balanceo,0.863000,0.529000,0.150800,0.234700,0.819500


---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 8. Sobremuestreo (Oversampling)

Aumenta la representacion de la clase minoritaria (diabeticos) generando nuevas muestras sinteticas o duplicando las existentes.

</div>

---

### 8.1 Random Oversampling
Duplica aleatoriamente muestras de la clase minoritaria. Simple pero puede sobreajustar.

In [41]:
ros_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("sampler", RandomOverSampler(random_state=SEED)),
    ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1))
])

metrics_ros = evaluate_model(ros_model, "Random Oversampling")

=== Random Oversampling ===



Classification Report

              precision    recall  f1-score   support

Sin diabetes       0.89      0.93      0.91     43667
    Diabetes       0.41      0.30      0.35      7069

    accuracy                           0.84     50736
   macro avg       0.65      0.62      0.63     50736
weighted avg       0.82      0.84      0.83     50736




METRICAS



Model,Accuracy,Precision,Recall,F1,ROC_AUC
Random Oversampling,0.842100,0.410200,0.304400,0.349500,0.788800


---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 9. Submuestreo (Undersampling)

Reduce el numero de muestras de la clase mayoritaria (no diabeticos) para equilibrar las clases. Riesgo: perder informacion relevante.

</div>

---

### 9.1 Random Undersampling
Elimina aleatoriamente muestras de la clase mayoritaria hasta equiparar las clases.

In [42]:
rus_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("sampler", RandomUnderSampler(random_state=SEED)),
    ("model", RandomForestClassifier(random_state=SEED, n_jobs=-1))
])

metrics_rus = evaluate_model(rus_model, "Random Undersampling")

=== Random Undersampling ===



Classification Report

              precision    recall  f1-score   support

Sin diabetes       0.95      0.69      0.80     43667
    Diabetes       0.29      0.77      0.42      7069

    accuracy                           0.71     50736
   macro avg       0.62      0.73      0.61     50736
weighted avg       0.86      0.71      0.75     50736




METRICAS



Model,Accuracy,Precision,Recall,F1,ROC_AUC
Random Undersampling,0.705500,0.290400,0.771300,0.421900,0.804200


---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 10. Combinaciones de Sobre y Submuestreo

Combina la generacion de muestras sinteticas con limpieza de la frontera, obteniendo datasets mas balanceados **y** menos ruidosos.

</div>

---

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 11. Metodos Basados en Ensambles

Integran el balanceo directamente dentro del algoritmo de clasificacion. Cada arbol o clasificador base recibe un subconjunto balanceado, sin necesidad de remuestrear el dataset completo.

</div>

---

### 11.1 Balanced Random Forest

**Idea:** para cada arbol del bosque, en vez de bootstrap del dataset completo, toma **todas** las muestras minoritarias y una muestra aleatoria **igual** de la mayoritaria.

| Clase | Dataset original | Dataset por arbol |
|---|---|---|
| Sin diabetes (0) | ~218,000 | ~35,300 |
| Diabetes (1) | ~35,300 | ~35,300 |

In [43]:
brf_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("model", BalancedRandomForestClassifier(
        n_estimators=300, random_state=SEED, n_jobs=-1,
        replacement=True, sampling_strategy="all"
    ))
])

metrics_brf = evaluate_model(brf_model, "Balanced Random Forest")

=== Balanced Random Forest ===



Classification Report

              precision    recall  f1-score   support

Sin diabetes       0.94      0.78      0.85     43667
    Diabetes       0.33      0.68      0.44      7069

    accuracy                           0.76     50736
   macro avg       0.63      0.73      0.65     50736
weighted avg       0.85      0.76      0.79     50736




METRICAS



Model,Accuracy,Precision,Recall,F1,ROC_AUC
Balanced Random Forest,0.764600,0.331300,0.677200,0.445000,0.811800


### 11.2 Easy Ensemble

Entrena multiples clasificadores AdaBoost, cada uno sobre un subconjunto balanceado diferente de la clase mayoritaria. La prediccion final es el promedio de todos los clasificadores.

In [ ]:
easy_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_no_scaled),
    ("model", EasyEnsembleClassifier(random_state=SEED, n_jobs=-1))
])

metrics_easy = evaluate_model(easy_model, "Easy Ensemble")

=== Easy Ensemble ===



Classification Report

              precision    recall  f1-score   support

Sin diabetes       0.95      0.72      0.82     43667
    Diabetes       0.31      0.77      0.44      7069

    accuracy                           0.73     50736
   macro avg       0.63      0.74      0.63     50736
weighted avg       0.86      0.73      0.77     50736




METRICAS



Model,Accuracy,Precision,Recall,F1,ROC_AUC
Easy Ensemble,0.729200,0.309300,0.765500,0.440600,0.820300


: 

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 12. Validacion Cruzada -- Comparacion de Modelos

Se aplica **Stratified K-Fold (k=5)** sobre los modelos seleccionados para estimar su capacidad de generalizacion. El estrato garantiza la misma proporcion de clases en cada pliegue.

</div>

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

scoring = ["accuracy", "precision", "recall", "f1", "roc_auc"]

baseline_model = ImbPipeline(steps=[
    ("preprocessor", preprocessor_scaled),
    ("model", LogisticRegression(max_iter=500, random_state=SEED))
])

models_cv = {
    "Baseline -- Sin balanceo": baseline_model,
    "Random Oversampling": ros_model,
    "Random Undersampling": rus_model,
    "Balanced RF": brf_model,
    "Easy Ensemble": easy_model,
}

results = []
for name, model in models_cv.items():
    print(f"  Evaluando: {name} ...", end=" ", flush=True)
    scores = cross_validate(model, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    results.append({
        "Modelo":     name,
        "Accuracy":   round(np.mean(scores["test_accuracy"]), 4),
        "Precision":  round(np.mean(scores["test_precision"]), 4),
        "Recall":     round(np.mean(scores["test_recall"]), 4),
        "F1":         round(np.mean(scores["test_f1"]), 4),
        "ROC_AUC":    round(np.mean(scores["test_roc_auc"]), 4),
    })
    print("OK")

results_df = pd.DataFrame(results).sort_values("F1", ascending=False)
display(results_df.style.hide(axis="index").background_gradient(
    subset=["Precision","Recall","F1","ROC_AUC"], cmap="RdYlGn"
))

  Evaluando: Baseline -- Sin balanceo ... OK
  Evaluando: Random Oversampling ... 

In [ ]:
# Grafica comparativa de F1
fig = px.bar(
    results_df,
    x="Modelo", y="F1", text="F1",
    title="Comparacion de Modelos -- F1-Score (Validacion Cruzada k=5)",
    template="simple_white",
    color="F1", color_continuous_scale="RdYlGn"
)
fig.update_traces(texttemplate="%{text:.3f}", textposition="outside")
fig.update_layout(
    xaxis_title="Modelo", yaxis_title="F1-Score",
    yaxis=dict(range=[0, 1]), title_x=0.5, height=500
)
fig.show()

# Grafica multimetrica
metrics_long = results_df.melt(
    id_vars="Modelo",
    value_vars=["Precision","Recall","F1","ROC_AUC"],
    var_name="Metrica", value_name="Valor"
)
fig2 = px.bar(
    metrics_long, x="Modelo", y="Valor", color="Metrica",
    barmode="group",
    title="Comparacion multimetrica por modelo",
    template="simple_white", height=500
)
fig2.update_layout(title_x=0.5, xaxis_tickangle=-30)
fig2.show()

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## 13. Ajuste del Umbral de Decision

El umbral por defecto es 0.5: si la probabilidad predicha es mayor a 0.5 el modelo clasifica como "diabetico". Subir el umbral exige mas confianza antes de clasificar positivo, lo que **aumenta Precision** pero reduce Recall. La curva Precision-Recall permite elegir el umbral optimo segun la prioridad clinica.

</div>

In [ ]:
# Seleccionar el mejor modelo de CV para ajustar umbral
best_model_name = results_df.iloc[0]["Modelo"]
best_model_pipe = models_cv[best_model_name]
best_model_pipe.fit(X_train, y_train)
y_prob_best = best_model_pipe.predict_proba(X_test)[:, 1]
y_pred_default = best_model_pipe.predict(X_test)

print(f"Mejor modelo: {best_model_name}")
print(f"\nUmbral 0.50 -- Precision: {precision_score(y_test, y_pred_default):.4f} | "
      f"Recall: {recall_score(y_test, y_pred_default):.4f} | "
      f"F1: {f1_score(y_test, y_pred_default):.4f}")

precisions, recalls, thresholds = precision_recall_curve(y_test, y_prob_best)
f1_per_thresh = 2 * (precisions * recalls) / (precisions + recalls + 1e-8)
best_idx = np.argmax(f1_per_thresh)
best_thresh = thresholds[best_idx]

y_pred_tuned = (y_prob_best >= best_thresh).astype(int)
print(f"Umbral {best_thresh:.2f}  -- Precision: {precision_score(y_test, y_pred_tuned):.4f} | "
      f"Recall: {recall_score(y_test, y_pred_tuned):.4f} | "
      f"F1: {f1_score(y_test, y_pred_tuned):.4f}")

In [ ]:
# Curva Precision-Recall interactiva
pr_df = pd.DataFrame({
    "Umbral": thresholds,
    "Precision": precisions[:-1],
    "Recall": recalls[:-1],
    "F1": f1_per_thresh[:-1]
})

fig = go.Figure()
fig.add_trace(go.Scatter(x=pr_df["Umbral"], y=pr_df["Precision"],
                          name="Precision", line=dict(color="#4C9BE8", width=2)))
fig.add_trace(go.Scatter(x=pr_df["Umbral"], y=pr_df["Recall"],
                          name="Recall", line=dict(color="#E87D4C", width=2)))
fig.add_trace(go.Scatter(x=pr_df["Umbral"], y=pr_df["F1"],
                          name="F1-Score", line=dict(color="#6CBF6C", width=2)))
fig.add_vline(x=best_thresh, line_dash="dash", line_color="black",
              annotation_text=f"Umbral optimo F1 = {best_thresh:.2f}")
fig.add_vline(x=0.5, line_dash="dot", line_color="gray",
              annotation_text="Default = 0.50")
fig.update_layout(
    title="Precision, Recall y F1 vs Umbral de Decision",
    xaxis_title="Umbral", yaxis_title="Metrica",
    template="simple_white", title_x=0.5, height=450
)
fig.show()

In [ ]:
# Curva Precision-Recall
fig = px.line(
    x=recalls, y=precisions,
    title=f"Curva Precision-Recall -- {best_model_name}",
    labels={"x": "Recall", "y": "Precision"},
    template="simple_white"
)
fig.add_scatter(
    x=[recalls[best_idx]], y=[precisions[best_idx]],
    mode="markers", marker=dict(size=12, color="red"),
    name=f"Umbral optimo ({best_thresh:.2f})"
)
fig.update_layout(title_x=0.5, height=450)
fig.show()

In [ ]:
# Matrices de confusion: umbral default vs ajustado
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, preds, title in zip(
    axes,
    [y_pred_default, y_pred_tuned],
    [f"Umbral default (0.50)", f"Umbral ajustado ({best_thresh:.2f})"]
):
    cm = confusion_matrix(y_test, preds)
    ConfusionMatrixDisplay(cm, display_labels=["Sin diabetes","Diabetes"]
    ).plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(title, fontweight="bold")
plt.suptitle(f"Impacto del ajuste de umbral -- {best_model_name}",
             fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 14. Importancia de Variables -- Modelo Final

In [ ]:
# Importancias del mejor modelo (si tiene feature_importances_)
try:
    importances = best_model_pipe.named_steps["model"].feature_importances_
    feature_names = all_features
    imp_df = pd.DataFrame({"Feature": feature_names, "Importance": importances})
    imp_df = imp_df.sort_values("Importance", ascending=False).head(21)

    fig = px.bar(
        imp_df.sort_values("Importance"),
        x="Importance", y="Feature",
        orientation="h",
        color="Importance", color_continuous_scale="Reds",
        title=f"Importancia de Variables -- {best_model_name}",
        template="plotly_dark"
    )
    fig.update_layout(height=700, title_x=0.5)
    fig.show()
except AttributeError:
    print(f"El modelo {best_model_name} no expone feature_importances_ directamente.")

---
## 15. Tabla Resumen y Seleccion del Modelo Final

In [ ]:
# Tabla completa con heatmap
print("=== Ranking de modelos por F1 (Validacion Cruzada k=5) ===")
display(
    results_df.style
    .hide(axis="index")
    .background_gradient(subset=["Precision","Recall","F1","ROC_AUC"], cmap="RdYlGn")
    .format({"Accuracy":"{:.4f}","Precision":"{:.4f}","Recall":"{:.4f}",
             "F1":"{:.4f}","ROC_AUC":"{:.4f}"})
)

best_row = results_df.iloc[0]
print(f"\nModelo seleccionado: {best_row['Modelo']}")
print(f"  F1       = {best_row['F1']:.4f}")
print(f"  Recall   = {best_row['Recall']:.4f}")
print(f"  Precision= {best_row['Precision']:.4f}")
print(f"  ROC_AUC  = {best_row['ROC_AUC']:.4f}")

### Criterios de seleccion

| Criterio | Consideracion |
|---|---|
| **F1-Score** | Metrica principal: equilibra Precision y Recall |
| **Recall** | Prioridad clinica: minimizar falsos negativos (diabeticos no detectados) |
| **ROC-AUC** | Capacidad discriminativa global, independiente del umbral |
| **Precision** | Reducir alarmas falsas innecesarias |
| **Accuracy** | **No** es criterio de seleccion en datos desbalanceados |

> **Nota:** el ajuste de umbral de la seccion 13 permite modificar el balance Precision/Recall del modelo ganador segun las necesidades del contexto clinico.

---
## 16. Entrenamiento Final y Serializacion

In [ ]:
# ========================================= #
# ENTRENAR MODELO FINAL CON TODOS LOS DATOS #
# ========================================= #

# Se usa el mejor modelo de CV, reentrenado sobre X completo
final_model = models_cv[best_model_name]
final_model.fit(X, y)
print(f"Modelo final entrenado: {best_model_name}")
print(f"Datos de entrenamiento: {X.shape[0]:,} muestras")

In [ ]:
# ==================== #
# GUARDAR MODELO FINAL #
# ==================== #

model_path    = MODELS_DIR / "model_classification.joblib"
features_path = MODELS_DIR / "features_classification.joblib"

joblib.dump(final_model,      model_path)
joblib.dump(X.columns.tolist(), features_path)

print(f"Modelo guardado en:    {model_path}")
print(f"Features guardadas en: {features_path}")

# Verificacion
loaded = joblib.load(model_path)
sample_proba = loaded.predict_proba(X_test.iloc[:5])[:, 1]
print(f"\nVerificacion -- probas primeras 5 muestras: {sample_proba.round(4)}")
print("Carga exitosa.")

---
<div style="text-align: justify; line-height: 1.5; font-size: 18px;">

## Resumen Ejecutivo

| Tecnica | Descripcion | Ventaja principal |
|---|---|---|
| **Random Oversampling** | Duplica muestras minoritarias | Simple, sin perder datos |
| **SMOTE** | Genera sinteticos interpolando vecinos | Reduce sobreajuste vs ROS |
| **ADASYN** | Enfoca generacion en zonas dificiles | Mejor frontera de decision |
| **Random Undersampling** | Elimina mayoritarios aleatoriamente | Muy rapido |
| **Tomek Links** | Limpia frontera de decision | Sin eliminar muestras utiles |
| **ENN** | Elimina mayoritarios mal clasificados | Frontera mas limpia que Tomek |
| **SMOTE + Tomek** | Genera sinteticos + limpia frontera | Balance y limpieza |
| **SMOTE + ENN** | Generacion + limpieza agresiva | Maxima limpieza de ruido |
| **Balanced Random Forest** | Balancea por arbol internamente | Robusto y eficiente |
| **Easy Ensemble** | Multiples AdaBoost balanceados | Excelente para alta dimension |
| **RUSBoost** | Undersampling + Boosting | Rapido y efectivo |
| **Balanced Bagging** | Bagging con bags balanceados | Reduce varianza |

**Metricas prioritarias para diabetes:** Recall > F1 > ROC-AUC > Precision > Accuracy

</div>

---

# Fin del Laboratorio 5

---